In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import RandomOverSampler
sns.set(style='whitegrid')
from dotenv import load_dotenv
import os
from pathlib import Path
import pickle

import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

load_dotenv()

True

In [2]:
project_root = Path.cwd().parent
data_path = project_root / os.getenv("DATA_PATH")
df = pd.read_csv(data_path)
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [3]:
channel_mean = df.groupby("Policy_Sales_Channel")["Response"].mean()
df["channel_target"] = df["Policy_Sales_Channel"].map(channel_mean)

region_mean = df.groupby("Region_Code")["Response"].mean()
df["region_target"] = df["Region_Code"].map(region_mean)

vehicle_age_map = {
    "< 1 Year": 0,
    "1-2 Year": 1,
    "> 2 Years": 2
}

df["Vehicle_Age_num"] = df["Vehicle_Age"].map(vehicle_age_map)

df["Age_x_Vehicle_Age"] = df["Age"] * df["Vehicle_Age_num"]

channel_freq = df["Policy_Sales_Channel"].value_counts()
df["Channel_freq"] = df["Policy_Sales_Channel"].map(channel_freq)

region_freq = df["Region_Code"].value_counts()
df["Region_Code_freq"] = df["Region_Code"].map(region_freq)

df["Age_x_Channel"] = df["Age"] * df["Channel_freq"]

df["Vehicle_Damage_bin"] = (df["Vehicle_Damage"] == "Yes").astype(int)

df["Damage_x_NotInsured"] = ((df["Vehicle_Damage"] == "Yes") &(df["Previously_Insured"] == 0)).astype(int)

df["Age_x_Region"] = df["Age"] * df["Region_Code_freq"]

df["Damage_Age"] = (df["Vehicle_Damage"] == "Yes") & (df["Vehicle_Age"] == "> 2 Years")

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  str    
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  str    
 7   Vehicle_Damage        381109 non-null  str    
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
 12  channel_target        381109 non-null  float64
 13  region_target         381109 non-null  float64
 14  Vehicle_Age_num       381109 non-null  int64  
 15  Age_x_Vehic

In [5]:
df.head(3)

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,...,region_target,Vehicle_Age_num,Age_x_Vehicle_Age,Channel_freq,Region_Code_freq,Age_x_Channel,Vehicle_Damage_bin,Damage_x_NotInsured,Age_x_Region,Damage_Age
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,...,0.187163,2,88,79700,106415,3506800,1,1,4682260,True
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,...,0.127662,1,76,79700,9251,6057200,0,0,703076,False
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,...,0.187163,2,94,79700,106415,3745900,1,1,5001505,True


In [6]:
df = df.astype({
    "Gender" : "category",
    "Driving_License" : "category",
    "Previously_Insured" : "category",
    "Response" : "category",
    "Vehicle_Damage_bin" : "category",
    "Damage_x_NotInsured" : "category"

})

In [7]:
df.drop(columns=['id','Age','Policy_Sales_Channel','Region_Code','Vehicle_Age','Vehicle_Damage'], axis=0, inplace=True)

In [8]:
cat_cols = df.select_dtypes(include=["object", "string", "category","bool"]).columns
num_cols = df.select_dtypes(exclude=["object", "string","category","bool"]).columns

In [9]:
print(f"Num Cols:\n{num_cols}\n")
print(f"Cat Cols:\n{cat_cols}")

Num Cols:
Index(['Annual_Premium', 'Vintage', 'channel_target', 'region_target',
       'Vehicle_Age_num', 'Age_x_Vehicle_Age', 'Channel_freq',
       'Region_Code_freq', 'Age_x_Channel', 'Age_x_Region'],
      dtype='str')

Cat Cols:
Index(['Gender', 'Driving_License', 'Previously_Insured', 'Response',
       'Vehicle_Damage_bin', 'Damage_x_NotInsured', 'Damage_Age'],
      dtype='str')


In [10]:
X = df.drop(columns=['Response'], axis=0)
y = df['Response']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [12]:
ohe_cols = cat_cols.drop(["Response"])
ohe_cols

Index(['Gender', 'Driving_License', 'Previously_Insured', 'Vehicle_Damage_bin',
       'Damage_x_NotInsured', 'Damage_Age'],
      dtype='str')

In [13]:
# One-hot encode categorical columns
ohe = OneHotEncoder(drop='first', sparse_output=False)

X_train_ohe = ohe.fit_transform(X_train[ohe_cols])
X_test_ohe = ohe.transform(X_test[ohe_cols])

# Numeric features
X_train_num = X_train[num_cols].to_numpy()
X_test_num = X_test[num_cols].to_numpy()

# Combine everything
X_train_final = np.hstack((X_train_num, X_train_ohe))
X_test_final = np.hstack((X_test_num, X_test_ohe))

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

In [16]:
number_of_class1 = df[df['Response'] == 1].shape[0]
number_of_class0 = df[df['Response'] == 0].shape[0]

dtc = DecisionTreeClassifier(max_depth = 5)
lrc = LogisticRegression(solver="liblinear",penalty="l1",class_weight="balanced")
rfc = RandomForestClassifier(n_estimators=200,class_weight="balanced",random_state=42)   
xgb  = XGBClassifier(n_estimators = 200, random_state = 2, scale_pos_weight = (number_of_class0 / number_of_class1))

clfs = {
    'DT': dtc,
    'LR': lrc,
    'RF': rfc,
    'xgb': xgb 
}

def train_classifier(clf, X_train_scaled, y_train, X_test_scaled, y_test):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    mean_cv_score = cv_scores.mean()
    clf.fit(X_train_scaled,y_train)
    y_pred = clf.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    return accuracy , precision, recall, f1, mean_cv_score

for name , clf in clfs.items():
    current_accuracy, current_precision, current_recall, current_f1, current_mean_cv_score = train_classifier(clf, X_train_scaled, y_train, X_test_scaled, y_test)
    print()
    print("For: ", name)
    print("Accuracy: ", current_accuracy)
    print("Precision: ", current_precision)
    print("Recall: ", current_recall)
    print("F1: ", current_f1)
    print("CV Score: ", current_mean_cv_score)


For:  DT
Accuracy:  0.877463199601165
Precision:  1.0
Recall:  0.00021408691928923143
F1:  0.0004280821917808219
CV Score:  0.8774365584977636

For:  LR
Accuracy:  0.692214846107423
Precision:  0.27682579829276005
Recall:  0.9372725326482552
F1:  0.42741384360050766
CV Score:  0.6893340804751033

For:  RF
Accuracy:  0.8634908556584713
Precision:  0.35386307396205663
Recall:  0.13776493256262043
F1:  0.1983203636643809
CV Score:  0.8642218247817205

For:  xgb
Accuracy:  0.7292251580908399
Precision:  0.2961933970773949
Recall:  0.8787197602226504
F1:  0.44304719756051486
CV Score:  0.7300344059249811


In [18]:
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

lr = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=22
)

param_grid = {
    "penalty": ["l1"],
    "C": [0.001, 0.01, 0.1, 1, 10],
    "solver": ["liblinear", "saga"]
}

grid = GridSearchCV(
    estimator=lr,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=skf,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

y_probs = best_model.predict_proba(X_test_scaled)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_probs)

optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print("Optimal Threshold:", optimal_threshold)

y_pred_opt = (y_probs >= optimal_threshold).astype(int)

accuracy = accuracy_score(y_test, y_pred_opt)
precision = precision_score(y_test, y_pred_opt,zero_division=0)
recall = recall_score(y_test, y_pred_opt)
f1 = f1_score(y_test, y_pred_opt)
roc_auc = roc_auc_score(y_test, y_probs)


print("\nFinal Model Performance")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", roc_auc)
print(confusion_matrix(y_test, y_pred_opt))
print(classification_report(y_test, y_pred_opt))

Fitting 10 folds for each of 10 candidates, totalling 100 fits
Best Parameters: {'C': 10, 'penalty': 'l1', 'solver': 'saga'}
Best CV ROC-AUC: 0.8471175041603706
Optimal Threshold: 0.5221792455631481

Final Model Performance
Accuracy: 0.7076565820891606
Precision: 0.2850546457163738
Recall: 0.9185399272104474
F1: 0.4350868297629611
ROC-AUC: 0.8486443461206734
[[45358 21522]
 [  761  8581]]
              precision    recall  f1-score   support

           0       0.98      0.68      0.80     66880
           1       0.29      0.92      0.44      9342

    accuracy                           0.71     76222
   macro avg       0.63      0.80      0.62     76222
weighted avg       0.90      0.71      0.76     76222



In [15]:
model_path = os.path.join(os.path.dirname(os.getcwd()), os.getenv("MODEL_PATH"))
os.makedirs(model_path, exist_ok=True)
with open(os.path.join(model_path,'model.pkl'), 'wb') as f:
    pickle.dump(best_model, f)

In [17]:
print(grid.best_params_)

{'C': 10, 'penalty': 'l1', 'solver': 'saga'}
